<b><font size="6" color="#E8800A">Week 9b · Regression trees</font></b><br>
<b><font size="4">Growing a tree, pruning it, and following one prediction</font></b><br>

A regression tree prices a car with a path of yes-or-no questions, each comparing one column with one threshold. The path ends in a leaf, and the training cars that landed in that leaf decide the price: their mean. How many questions the tree may ask, and how large a leaf must be, set how closely it follows the training cars.

<div class="alert alert-block alert-info">

# TOC<a class="anchor" id="toc"></a>
* [<font color='#E8800A'>How a tree chooses a split</font>](#split)
* [<font color='#E8800A'>Fitting a tree</font>](#fit)
* [<font color='#E8800A'>How a tree predicts</font>](#predict)
* [<font color='#E8800A'>What a tree cannot do</font>](#limits)
* [<font color='#E8800A'>The settings of a tree</font>](#settings)
* [<font color='#E8800A'>Two ways to prune</font>](#prune)
* [<font color='#E8800A'>Nested cross-validation</font>](#nested)
* [<font color='#E8800A'>The grid search</font>](#search)
* [<font color='#E8800A'>The number we stand behind</font>](#board)
* [<font color='#E8800A'>Key takeaways</font>](#takeaways)
* [<font color='#E8800A'>References</font>](#references)

</div>

# <font color='#E8800A'>How a tree chooses a split</font> <a class="anchor" id="split"></a>
[Back to TOC](#toc)

A tree is built one question at a time. Each question splits the cars in a node into two groups, chosen so that the prices in each group lie closer to the group's own mean than to the node's. A toy example of eight cars, small enough to compute every quantity, shows how the first question is chosen, and a fitted tree is drawn beside it.

In [ ]:
from tqdm.auto import tqdm

%pip install -q --disable-pip-version-check supertree==1.1.0

import sys
from pathlib import Path

# course_helpers.py and preprocessing.py sit in this folder, next to the notebook.
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import ParameterGrid, cross_validate, train_test_split
from sklearn.tree import DecisionTreeRegressor, plot_tree
from supertree import SuperTree

from course_helpers import (
    PLOT_BLUE,
    PLOT_ORANGE,
    Scoreboard,
    record,
    result_context,
    results_before,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 150
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats("svg")
random_state = 33
np.random.seed(random_state)  # For reproducibility

# Fold-safe encoding means a rare level can occur only in validation, so this
# message would repeat once per scored fold.
warnings.filterwarnings(
    "ignore",
    message="Found unknown categories.*during transform",
    category=UserWarning,
)

from preprocessing import (
    CachedSearchCV,
    FoldPreprocessor,
    PreparedEstimator,
    load_recipe,
    outer_folds,
    outer_splitter,
    prepared_folds,
)
regression_recipe_log = "../../logs/week_04_feature_work_regression_log.json"


__Step 1:__ Write down eight toy cars: age in years, mileage in thousand kilometres, and price in thousand euros.

In [ ]:
toy = pd.DataFrame([[1, 40, 24], [2, 20, 22], [3, 100, 23], [4, 60, 14], [5, 80, 17], [6, 140, 9], [7, 120, 12], [8, 160, 8]],
                   columns=["age", "mileage", "price"])
toy.T

A regression tree predicts the mean price of the cars in a leaf, and measures how well a group is described by its mean with the **squared-error impurity**

$$I = \frac{1}{n}\sum_{i=1}^{n} (y_i - \bar{y})^2,$$

where $n$ is the number of cars in the group, $i$ runs over them, $y_i$ is the price of car $i$ and $\bar{y}$ is the mean price of the group. The mean is the single price with the smallest squared error on the group, and $I$ is that smallest value, in squared units of price.

__Step 2:__ The squared-error impurity of the eight cars: their mean price, then the mean squared deviation from it. Type the impurity as `impurity_parent = ((toy["price"] - toy["price"].mean()) ** 2).mean()`.

In [ ]:
impurity_parent = ...
print(f"mean price: {toy['price'].mean():.3f}")
print(f"squared-error impurity of the eight cars: {impurity_parent:.3f}")

A split sends the cars whose value in one column is at most a threshold to the left child $L$, and the others to the right child $R$. Its **impurity decrease** is

$$\Delta = I_P - \frac{n_L}{n_P}\,I_L - \frac{n_R}{n_P}\,I_R,$$

where $P$ is the group being split, $n_P$, $n_L$ and $n_R$ are the numbers of cars in the group and in its two children, and $I_P$, $I_L$ and $I_R$ are their impurities. The weights $n_L / n_P$ and $n_R / n_P$ add to 1, so the last two terms are the children's impurity averaged by size. The tree tries every threshold of every column, taking the midpoint between neighbouring values, and keeps the split with the largest $\Delta$.

__Step 3:__ Every candidate threshold of both columns: the cars each side receives, the impurity of each side and the impurity decrease. Type the two side impurities as `impurity_left = ((left - left.mean()) ** 2).mean()` and `impurity_right = ((right - right.mean()) ** 2).mean()`.

In [ ]:
candidates = []
for column in ["age", "mileage"]:
    values = np.sort(toy[column].unique())
    for threshold in (values[:-1] + values[1:]) / 2:
        left, right = toy["price"][toy[column] <= threshold], toy["price"][toy[column] > threshold]
        impurity_left = ...
        impurity_right = ...
        weighted = (len(left) * impurity_left + len(right) * impurity_right) / len(toy)
        candidates.append({"column": column, "threshold": threshold, "left": len(left), "right": len(right),
                           "impurity_left": impurity_left, "impurity_right": impurity_right,
                           "weighted": weighted, "decrease": impurity_parent - weighted})
candidates = pd.DataFrame(candidates).sort_values("decrease", ascending=False)
candidates.head(5).round(3)

**The best candidate is `age <= 3.5`, with an impurity decrease of 28.359: it sends 3 cars left at an impurity of 0.667 and 5 right at 10.800, which weight to 7.000, against the parent's 35.359.** The next candidates, `age <= 5.5` and `mileage <= 110.0`, send the same 5 and 3 cars the other way and tie at 25.026.

__Step 4:__ Fit a tree limited to two questions on the same eight cars: `DecisionTreeRegressor(max_depth=2, random_state=random_state)`.

In [ ]:
toy_tree = ...
toy_tree = toy_tree.fit(toy[["age", "mileage"]], toy["price"])

__Step 5:__ Draw the toy tree.

In [ ]:
figure, axis = plt.subplots(figsize=(10, 5))
plot_tree(toy_tree, feature_names=["age", "mileage"], filled=True, rounded=True, precision=2,
          fontsize=11, ax=axis)
plt.show()

**How to read a box.** A box that has children starts with its question: cars whose value is at most the threshold go left, the others go right, and a leaf has no question. `squared_error` is the mean squared deviation of the prices of the cars that reach the box from their mean price, in squared thousand euros, so a smaller number means more alike cars. `samples` is how many training cars reach the box, and `value` is the mean price of those cars, which is the prediction when the box is a leaf. The colour follows `value`, darker for a higher price, so it shows the prediction and not how alike the cars are. The root reads `age <= 3.5`, `squared_error = 35.36`, `samples = 8` and `value = 16.12`: the eight prices add to 129, their mean is 129 / 8 = 16.125, and their mean squared deviation is 282.875 / 8 = 35.359, the impurity computed above.

__Step 6:__ The first question of the fitted tree, beside the best candidate of the table.

In [ ]:
best_candidate = candidates.iloc[0]
print(f"by hand: {best_candidate['column']} <= {best_candidate['threshold']}, impurity decrease {best_candidate['decrease']:.3f}")
print(f"fitted:  {['age', 'mileage'][toy_tree.tree_.feature[0]]} <= {toy_tree.tree_.threshold[0]},"
      f" impurity {toy_tree.tree_.impurity[0]:.3f}, then {toy_tree.tree_.impurity[toy_tree.tree_.children_left[0]]:.3f}"
      f" and {toy_tree.tree_.impurity[toy_tree.tree_.children_right[0]]:.3f}")

**The fitted tree asks `age <= 3.5` first, the table's best candidate, and its impurities, 35.359, 0.667 and 10.800, are the table's.** The three cars on the left ask a second question about `age`, and the five on the right one about `mileage`.

__Step 7:__ Fit a tree with one question on the cubes of the two columns, and compare its groups with the first question's.

In [ ]:
cubed_tree = DecisionTreeRegressor(max_depth=1, random_state=random_state).fit(toy[["age", "mileage"]] ** 3, toy["price"])
plain_tree = DecisionTreeRegressor(max_depth=1, random_state=random_state).fit(toy[["age", "mileage"]], toy["price"])
print(f"cubed question: {['age', 'mileage'][cubed_tree.tree_.feature[0]]} <= {cubed_tree.tree_.threshold[0]}, in cubed units")
print(f"same groups of cars: {np.array_equal(plain_tree.apply(toy[['age', 'mileage']]), cubed_tree.apply(toy[['age', 'mileage']] ** 3))}")

**The cubed columns give the same groups.** The question moves to `age <= 45.5` in cubed units: 45.5 is the midpoint of the cubes 27 and 64, not the cube of 3.5, and the groups of cars are the same. A tree reads the order of a column's values, so a strictly increasing change of a column, such as the log or the scaling of the recipe, leaves the training groups unchanged. The threshold is the midpoint of two neighbouring training values, and a nonlinear change moves that midpoint, so a new value between those two could change sides.

# <font color='#E8800A'>Fitting a tree</font> <a class="anchor" id="fit"></a>
[Back to TOC](#toc)

Fitting grows the tree on the prepared training cars: each node takes the split with the largest impurity reduction, and stops splitting when no column can separate its cars or a setting forbids another split. The default tree has no setting that forbids one.

__Step 8:__ Load the regression spine and split it.

In [ ]:
data, regression_recipe = load_recipe(
    "../../data/interim/cars4you.csv",
    regression_recipe_log,
)
features = data.loc[:, regression_recipe["categorical"] + regression_recipe["numeric"]].copy()
target = data["price"]

X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=random_state,
)
print(f"{len(X_train):,} training cars, {len(X_test):,} test cars")

__Step 9:__ Prepare the cars, and fix the folds every score is read on.

In [ ]:
tree_prep = FoldPreprocessor(regression_recipe).fit(X_train, y_train)
Z_train = tree_prep.transform(X_train)
Z_test = tree_prep.transform(X_test)
prepared_names = list(tree_prep.get_feature_names_out())

# The ten folds every validation score is read on.
tree_folds = outer_splitter("regression", random_state=random_state)
tree_prepared = prepared_folds(
    FoldPreprocessor(regression_recipe), tree_folds, X_train, y_train, n_jobs=4)
print(f"{Z_train.shape[1]} prepared columns, {tree_folds.get_n_splits()} folds")

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html'>sklearn.tree.DecisionTreeRegressor(*, criterion='squared_error', splitter='best', max_depth=None, min_samples_split=2, min_samples_leaf=1, min_weight_fraction_leaf=0.0, max_features=None, random_state=None, max_leaf_nodes=None, min_impurity_decrease=0.0, ccp_alpha=0.0, monotonic_cst=None)</a>

**Definition:**
A binary tree that predicts a row's target as the mean of the training rows in its leaf.

**Main Parameters:**
- `max_depth` (default `None`): the most questions on any path.
- `min_samples_leaf` (default `1`): the fewest training rows a leaf may hold.
- `min_samples_split` (default `2`): the fewest training rows a node needs to ask another question.
- `criterion` (default `'squared_error'`): the impurity a split must lower; `'absolute_error'` predicts leaf medians instead of means.
- `max_features` (default `None`): how many columns each split considers.
- `ccp_alpha` (default `0.0`): the cost of a leaf in cost-complexity pruning.
- `random_state`: fixes the order in which columns are tried, which decides ties.

**Main Attributes (after fitting):**
- `tree_`: the fitted tree: for each node its children, column, threshold, training rows, impurity and mean target.
- `feature_importances_`: each column's total impurity reduction, scaled to sum to 1.

**Main Methods:**
- `fit(X, y)`: grows the tree.
- `apply(X)`: the leaf each row reaches.
- `decision_path(X)`: the nodes each row passes through.
- `predict(X)`: the mean target of the training rows in the leaf.
- `get_depth()` and `get_n_leaves()`: the size of the fitted tree.
- `cost_complexity_pruning_path(X, y)`: the alphas at which branches would be pruned.
</div>

__Step 10:__ Score `DecisionTreeRegressor()` on the ten folds, and open the log.

In [ ]:
tree_log = []
default_model = DecisionTreeRegressor(random_state=random_state)
default_trees = [clone(default_model).fit(fold["X_fit"], fold["y_fit"]) for fold in tqdm(tree_prepared, desc='tree prepared', leave=True, position=0)]
default_cv = {"train": [mean_absolute_error(fold["y_fit"], tree.predict(fold["X_fit"]))
                        for fold, tree in zip(tree_prepared, default_trees)],
              "validation": [mean_absolute_error(fold["y_score"], tree.predict(fold["X_score"]))
                             for fold, tree in zip(tree_prepared, default_trees)]}
default_fitted = clone(default_model).fit(Z_train, y_train)

In [ ]:
record(tree_log, {"frame": type(tree_folds).__name__, "rows": len(y_train),
                  "folds": tree_folds.get_n_splits(), "name": "DecisionTreeRegressor()",
                  "train": default_cv["train"], "validation": default_cv["validation"],
                  "model": default_fitted},
       Z_test, y_test, metric=mean_absolute_error, label="MAE")

**The default tree has a training MAE of 0.66 EUR, a validation MAE of 3,053.52 EUR and a test MAE of 2,831.21 EUR.** It keeps asking questions until almost every training car has a leaf of its own, so it nearly reproduces the training prices, and the 3,052.86 EUR between the first two scores is how far it follows the training cars beyond what carries to new ones.

__Step 11:__ Draw the whole default tree, every node, in a zoomable viewer whose footer states its nodes, leaves and depth.

In [ ]:
SuperTree(default_fitted, Z_train, y_train.to_numpy(), prepared_names, None).show_tree(start_depth=default_fitted.get_depth(), max_samples=500)

**Drawn in full, the default tree has 28 levels and 3,161 leaves, too many boxes to read.** Its training MAE of 0.66 EUR means this whole structure reproduces the training prices almost exactly.

__Step 12:__ Score the tree limited to two questions on the same folds, and print its scores beside the default's.

In [ ]:
small_model = DecisionTreeRegressor(max_depth=2, random_state=random_state)
small_trees = [clone(small_model).fit(fold["X_fit"], fold["y_fit"]) for fold in tqdm(tree_prepared, desc='tree prepared', leave=True, position=0)]
small_cv = {"train": [mean_absolute_error(fold["y_fit"], tree.predict(fold["X_fit"]))
                      for fold, tree in zip(tree_prepared, small_trees)],
            "validation": [mean_absolute_error(fold["y_score"], tree.predict(fold["X_score"]))
                           for fold, tree in zip(tree_prepared, small_trees)]}
small_tree = clone(small_model).fit(Z_train, y_train)

In [ ]:
print(f"depth two: training MAE {np.mean(small_cv['train']):.4f}, validation MAE {np.mean(small_cv['validation']):.4f}")
print(f"default:   training MAE {np.mean(default_cv['train']):.4f}, validation MAE {np.mean(default_cv['validation']):.4f}")

__Step 13:__ Read the depth and the number of leaves of the two fitted trees.

In [ ]:
pd.DataFrame([{"tree": "default", "depth": default_fitted.get_depth(), "leaves": default_fitted.get_n_leaves()},
              {"tree": "depth two", "depth": small_tree.get_depth(), "leaves": small_tree.get_n_leaves()}]).set_index("tree")

**The default tree is 28 questions deep with 3,161 leaves, and the depth-two tree has 4 leaves.** The depth-two tree has a training MAE of 4,714.70 EUR and a validation MAE of 4,731.69: the two are close, but both are far above the default's validation MAE of 3,053.52, so two questions are too few for prices.

# <font color='#E8800A'>How a tree predicts</font> <a class="anchor" id="predict"></a>
[Back to TOC](#toc)

A fitted tree is a set of nested questions, so it can be drawn in full. The depth-two tree below has four leaves, and the steps that follow take the first test car from the top question to its leaf and to its prediction.

__Step 14:__ Draw the depth-two tree fitted on all the training rows, in the zoomable viewer.

In [ ]:
SuperTree(small_tree, Z_train, y_train.to_numpy(), prepared_names, None).show_tree(start_depth=small_tree.get_depth(), max_samples=500)

**The top question is on `transmission_Manual` at 0.5, and both children then ask about `year`, at 0.9136 on the left and 0.9182 on the right.** Each question box scatters the prices of the cars that reach it against its column, with a dashed line at the threshold and at the mean price of each side, and each leaf states its target, the mean price, and its samples. The four leaves predict 17,640.58, 29,974.51, 10,560.09 and 16,290.86 EUR from 833, 595, 1,353 and 419 training cars.

__Step 15:__ Follow the first test car through the tree: at each question, its prepared value, the threshold and the branch taken.

In [ ]:
example = Z_test[:1]
trace = []
for node in small_tree.decision_path(example).indices:
    column = small_tree.tree_.feature[node]
    if column >= 0:
        trace.append({"node": node, "column": prepared_names[column], "value": example[0, column],
                      "threshold": small_tree.tree_.threshold[node],
                      "branch": "left" if example[0, column] <= small_tree.tree_.threshold[node] else "right"})
leaf = small_tree.apply(example)[0]
print(pd.DataFrame(trace).round(4).to_string(index=False))
print(f"leaf {leaf}, which holds {small_tree.tree_.n_node_samples[leaf]} training cars")

__Step 16:__ The mean price of the leaf's training cars: the prediction by hand.

In [ ]:
in_leaf = small_tree.apply(Z_train) == leaf
print(f"mean of the {in_leaf.sum()} training prices in the leaf: {y_train.to_numpy()[in_leaf].mean():,.2f} EUR")

__Step 17:__ `predict` from the fitted tree, beside the car's price.

In [ ]:
print(f"predict: {small_tree.predict(example)[0]:,.2f} EUR")
print(f"the car's price: {y_test.iloc[0]:,.2f} EUR")

**The first test car goes right at `transmission_Manual`, whose value 1.0000 is above 0.5, and left at `year`, whose value 0.7727 is at most 0.9182, and reaches the leaf of 1,353 training cars.** The mean of their prices, 10,560.09 EUR, is the prediction, equal to `predict`. The car's price was 13,500.00 EUR, 2,939.91 above it.

# <font color='#E8800A'>What a tree cannot do</font> <a class="anchor" id="limits"></a>
[Back to TOC](#toc)

A leaf prices every car that reaches it in the same way. Three consequences are read from the default tree: how few different prices it can give, what it does with a car unlike any it was fitted on, and what its column importances measure.

__Step 18:__ Count the different prices each tree predicts for the test cars.

In [ ]:
default_prices = default_fitted.predict(Z_test)
small_prices = small_tree.predict(Z_test)
print(f"default: {len(np.unique(default_prices))} different prices for {len(default_prices):,} test cars")
print(f"depth two: {len(np.unique(small_prices))} different prices for {len(small_prices):,} test cars")

__Step 19:__ Count the training cars in each leaf of the default tree.

In [ ]:
leaf_sizes = default_fitted.tree_.n_node_samples[default_fitted.tree_.children_left == -1]
print(f"{len(leaf_sizes)} leaves; median {np.median(leaf_sizes):.0f} training cars per leaf;"
      f" {np.mean(leaf_sizes == 1):.4f} of the leaves hold a single car")

**The default tree predicts 493 different prices for the 800 test cars, and the depth-two tree 4.** The default's 3,161 leaves hold a median of 1 training car, and 0.9880 of them hold a single car, so a predicted price is very often one training car's price.

__Step 20:__ Build a probe unlike any training car: the first one, with every numeric column set to ten times its largest training value.

In [ ]:
probe = X_train.iloc[[0]].copy()
numeric = regression_recipe["numeric"]
probe.loc[:, numeric] = 10 * X_train[numeric].max().to_numpy()
probe_ready = tree_prep.transform(probe)

__Step 21:__ Ask the default tree to price the probe, beside the range of the training prices.

In [ ]:
print(f"predicted price: {default_fitted.predict(probe_ready)[0]:,.2f} EUR")
print(f"training prices: {y_train.min():,.0f} to {y_train.max():,.0f} EUR")
print(f"training cars in its leaf: {default_fitted.tree_.n_node_samples[default_fitted.apply(probe_ready)[0]]}")

**The probe, with every numeric column ten times its largest training value, is priced at 15,680.00 EUR by a leaf of 1 training car, inside the training prices of 650 to 145,000 EUR.** A regression tree answers with training prices, so it cannot price outside their range, and nothing in the answer shows that the probe resembles no training car.

__Step 22:__ Read the five columns with the largest `feature_importances_` in the default tree.

In [ ]:
importance = pd.Series(default_fitted.feature_importances_, index=prepared_names).nlargest(5)
importance.round(4)

__Step 23:__ Read the number of training cars, their mean price and their mean year, by brand.

In [ ]:
training_cars = X_train.assign(price=y_train)
by_brand = training_cars.groupby("Brand").agg(cars=("price", "size"), mean_price=("price", "mean"),
                                             mean_year=("year", "mean")).sort_values("mean_price")
by_brand.round(2)

**`transmission_Manual` carries 0.2846 of the default tree's total impurity reduction, `engineSize` 0.2721 and `year` 0.2467.** These numbers say how much each column lowered impurity in this tree on these training cars, not what changes a price. The brand table shows the same caution: mean prices run from 10,221.01 EUR for Opel to 24,092.69 for Mercedes, but the mean year also differs, from 2,016.79 for Toyota to 2,017.43 for Skoda. A brand groups cars of different ages, so its price gap is an association and not the price change from changing a brand.

# <font color='#E8800A'>The settings of a tree</font> <a class="anchor" id="settings"></a>
[Back to TOC](#toc)

Eight settings move here, each alone with the others at their defaults: `max_depth`, `min_samples_leaf`, `min_samples_split`, `splitter`, `criterion`, `max_features`, `ccp_alpha` and the scale of the target. Each step prints training, validation and test MAE side by side, and the validation score alone picks the setting that joins the log; the tree of each logged setting is drawn beside the default's. One grid over the combinations makes the final choice, after nested cross-validation.

**`max_depth`: how many questions a path may ask.** A tree with `max_depth=d` asks at most `d` questions about a car before it prices it, so it has at most $2^d$ leaves. `None` lets every branch grow until its cars cannot be separated further. A shallow tree gives unlike cars the same price, and a deep one gives almost every training car a price of its own.

**Prediction:** training MAE falls toward 0 with depth, while validation MAE falls, then rises.

__Step 24:__ Set the values of `max_depth` to try: `[3, 5, 8, 12, None]`. At `None` it is at its default.

In [ ]:
max_depth_values = [3, 5, 8, 12, None]

In the loop, the model is `DecisionTreeRegressor(max_depth=value, random_state=random_state)`, and the sweep prints training, validation and test MAE for each value.

In [ ]:
max_depth_rows = []
for value in tqdm(max_depth_values, desc='max depth values', leave=True, position=0):
    model = ...
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in tree_prepared]
    fitted = clone(model).fit(Z_train, y_train)
    row = {"configuration": f"DecisionTreeRegressor(max_depth={value!r})", "setting": str(value),
           "train": [mean_absolute_error(fold["y_fit"], tree.predict(fold["X_fit"]))
                     for fold, tree in zip(tree_prepared, fold_models)],
           "validation": [mean_absolute_error(fold["y_score"], tree.predict(fold["X_score"]))
                          for fold, tree in zip(tree_prepared, fold_models)],
           "model": fitted}
    row["train_MAE"], row["cv_MAE"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_MAE"] = mean_absolute_error(y_test, fitted.predict(Z_test))
    row["depth"], row["leaves"] = fitted.get_depth(), fitted.get_n_leaves()
    max_depth_rows.append(row)
max_depth_sweep = pd.DataFrame(max_depth_rows)
max_depth_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "depth", "leaves"]].round(2)

In [ ]:
# The sweep's winner joins the log.
best = max_depth_sweep.loc[max_depth_sweep["cv_MAE"].idxmin()]
record(tree_log, {"frame": type(tree_folds).__name__, "rows": len(y_train),
                  "folds": tree_folds.get_n_splits(), "name": best["configuration"],
                  "train": best["train"], "validation": best["validation"], "model": best["model"]},
       Z_test, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

In [ ]:
SuperTree(best['model'], Z_train, y_train.to_numpy(), prepared_names, None).show_tree(start_depth=best['model'].get_depth(), max_samples=500)

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(max_depth_sweep["depth"], max_depth_sweep["train_MAE"], marker="o", markersize=8, linewidth=2,
         color=PLOT_ORANGE, label="training (ten folds)")
plt.plot(max_depth_sweep["depth"], max_depth_sweep["cv_MAE"], marker="o", markersize=8, linewidth=2,
         color=PLOT_BLUE, label="validation (ten folds)")
plt.xlabel("depth of the tree fitted on all the training rows")
plt.ylabel("MAE (EUR)")
plt.title("Training and validation MAE against tree depth")
plt.legend()
plt.show()

**Training MAE falls from 4,012.16 EUR at depth 3 to 0.66 with no limit, while validation MAE is lowest at `max_depth=12`, 2,925.67, and rises to 3,053.52 for the unrestricted tree.** `DecisionTreeRegressor(max_depth=12)` joins the log, with a test MAE of 2,735.71.

**`min_samples_leaf`: the smallest group that may set a price.** A split is allowed only if both children keep at least this many training cars, so a larger value makes every price a mean of more cars and stops the tree from isolating single ones.

**Prediction:** a larger leaf size raises training MAE and lowers validation MAE, until the leaves become too coarse.

__Step 25:__ Set the values of `min_samples_leaf` to try: `[1, 5, 20, 50]`. At `1` it is at its default.

In [ ]:
min_samples_leaf_values = ...

In the loop, the model is `DecisionTreeRegressor(min_samples_leaf=value, random_state=random_state)`.

In [ ]:
min_samples_leaf_rows = []
for value in tqdm(min_samples_leaf_values, desc='min samples leaf values', leave=True, position=0):
    model = DecisionTreeRegressor(min_samples_leaf=value, random_state=random_state)
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in tree_prepared]
    fitted = clone(model).fit(Z_train, y_train)
    row = {"configuration": f"DecisionTreeRegressor(min_samples_leaf={value!r})", "setting": str(value),
           "train": [mean_absolute_error(fold["y_fit"], tree.predict(fold["X_fit"]))
                     for fold, tree in zip(tree_prepared, fold_models)],
           "validation": [mean_absolute_error(fold["y_score"], tree.predict(fold["X_score"]))
                          for fold, tree in zip(tree_prepared, fold_models)],
           "model": fitted}
    row["train_MAE"], row["cv_MAE"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_MAE"] = mean_absolute_error(y_test, fitted.predict(Z_test))
    row["depth"], row["leaves"] = fitted.get_depth(), fitted.get_n_leaves()
    min_samples_leaf_rows.append(row)
min_samples_leaf_sweep = pd.DataFrame(min_samples_leaf_rows)
min_samples_leaf_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "depth", "leaves"]].round(2)

In [ ]:
# The sweep's winner joins the log.
best = min_samples_leaf_sweep.loc[min_samples_leaf_sweep["cv_MAE"].idxmin()]
record(tree_log, {"frame": type(tree_folds).__name__, "rows": len(y_train),
                  "folds": tree_folds.get_n_splits(), "name": best["configuration"],
                  "train": best["train"], "validation": best["validation"], "model": best["model"]},
       Z_test, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

In [ ]:
SuperTree(best['model'], Z_train, y_train.to_numpy(), prepared_names, None).show_tree(start_depth=best['model'].get_depth(), max_samples=500)

**Validation MAE is lowest at `min_samples_leaf=5`, 2,795.38 EUR, against 3,053.52 at the default of 1, and rises to 2,890.55 at 20 and 3,184.70 at 50.** Training MAE rises from 0.66 to 1,769.28, 2,524.06 and 2,998.73, and the leaves fall from 3,161 to 521, 123 and 50. `DecisionTreeRegressor(min_samples_leaf=5)` joins the log.

**`min_samples_split`: the smallest group that may ask another question.** A node with fewer training cars than this is a leaf. It limits the parent, where `min_samples_leaf` limits the children: with `min_samples_leaf=1`, a node of 50 cars can still split into one car and 49.

**Prediction:** a larger value gives a smaller tree and a higher training MAE.

__Step 26:__ Set the values of `min_samples_split` to try: `[2, 10, 50]`. At `2` it is at its default.

In [ ]:
min_samples_split_values = [2, 10, 50]

In [ ]:
min_samples_split_rows = []
for value in tqdm(min_samples_split_values, desc='min samples split values', leave=True, position=0):
    model = DecisionTreeRegressor(min_samples_split=value, random_state=random_state)
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in tree_prepared]
    fitted = clone(model).fit(Z_train, y_train)
    row = {"configuration": f"DecisionTreeRegressor(min_samples_split={value!r})", "setting": str(value),
           "train": [mean_absolute_error(fold["y_fit"], tree.predict(fold["X_fit"]))
                     for fold, tree in zip(tree_prepared, fold_models)],
           "validation": [mean_absolute_error(fold["y_score"], tree.predict(fold["X_score"]))
                          for fold, tree in zip(tree_prepared, fold_models)],
           "model": fitted}
    row["train_MAE"], row["cv_MAE"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_MAE"] = mean_absolute_error(y_test, fitted.predict(Z_test))
    row["depth"], row["leaves"] = fitted.get_depth(), fitted.get_n_leaves()
    min_samples_split_rows.append(row)
min_samples_split_sweep = pd.DataFrame(min_samples_split_rows)
min_samples_split_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "depth", "leaves"]].round(2)

In [ ]:
# The sweep's winner joins the log.
best = min_samples_split_sweep.loc[min_samples_split_sweep["cv_MAE"].idxmin()]
record(tree_log, {"frame": type(tree_folds).__name__, "rows": len(y_train),
                  "folds": tree_folds.get_n_splits(), "name": best["configuration"],
                  "train": best["train"], "validation": best["validation"], "model": best["model"]},
       Z_test, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

In [ ]:
SuperTree(best['model'], Z_train, y_train.to_numpy(), prepared_names, None).show_tree(start_depth=best['model'].get_depth(), max_samples=500)

**`min_samples_split=10` gives the lowest validation MAE of its sweep, 2,838.23 EUR, against 3,053.52 at 2 and 2,908.71 at 50.** The trees keep 720 and 145 leaves at 10 and 50. `DecisionTreeRegressor(min_samples_split=10)` joins the log.

**`splitter`: how the threshold is found.** `'best'` tries every threshold of every column the split may consider, as in the toy table above, and keeps the largest impurity decrease. `'random'` draws one random threshold for each of those columns and keeps the best of the draws. It searches less, so it is faster, and the tree it grows depends on `random_state` and is usually less accurate.

**Prediction:** `'random'` raises validation MAE for a single tree.

__Step 27:__ Set the splitters to try: `["best", "random"]`. `"best"` is the default.

In [ ]:
splitter_values = ["best", "random"]

In [ ]:
splitter_rows = []
for value in tqdm(splitter_values, desc='splitter values', leave=True, position=0):
    model = DecisionTreeRegressor(splitter=value, random_state=random_state)
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in tree_prepared]
    fitted = clone(model).fit(Z_train, y_train)
    row = {"configuration": f"DecisionTreeRegressor(splitter={value!r})", "setting": str(value),
           "train": [mean_absolute_error(fold["y_fit"], tree.predict(fold["X_fit"]))
                     for fold, tree in zip(tree_prepared, fold_models)],
           "validation": [mean_absolute_error(fold["y_score"], tree.predict(fold["X_score"]))
                          for fold, tree in zip(tree_prepared, fold_models)],
           "model": fitted}
    row["train_MAE"], row["cv_MAE"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_MAE"] = mean_absolute_error(y_test, fitted.predict(Z_test))
    row["depth"], row["leaves"] = fitted.get_depth(), fitted.get_n_leaves()
    splitter_rows.append(row)
splitter_sweep = pd.DataFrame(splitter_rows)
splitter_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "depth", "leaves"]].round(2)

In [ ]:
# The sweep's winner joins the log.
best = splitter_sweep.loc[splitter_sweep["cv_MAE"].idxmin()]
record(tree_log, {"frame": type(tree_folds).__name__, "rows": len(y_train),
                  "folds": tree_folds.get_n_splits(), "name": best["configuration"],
                  "train": best["train"], "validation": best["validation"], "model": best["model"]},
       Z_test, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

**`'random'` raises validation MAE to 3,106.17 EUR against 3,053.52 for `'best'`, as predicted, with a similar tree, 3,168 leaves against 3,161.** Its test MAE is 3,066.98 against 2,831.21. Validation selects, so the default wins and `DecisionTreeRegressor(splitter='best')` joins the log with the default's scores, and its tree is the default tree drawn above.

**`criterion`: the impurity a split must lower.** `'squared_error'` is the impurity of the toy example above, and each leaf predicts its mean. `'friedman_mse'` is an older option that scikit-learn now maps to `'squared_error'`, so it is not swept. `'absolute_error'` measures each group by its mean absolute deviation from its median, in euros, and each leaf predicts its median. In the drawn boxes `absolute_error` then takes the place of `squared_error`, and `value` is the median of the prices. `'poisson'` measures half the Poisson deviance of the prices in the group, which suits non-negative counts and lets the spread grow with the mean.

**Prediction:** each criterion grows a different tree, and `'absolute_error'` lowers validation MAE because it fits the quantity MAE reports.

__Step 28:__ The right child of the toy example's best split, under two criteria: its squared-error impurity, and its absolute error around the median.

In [ ]:
right_prices = toy["price"][toy[best_candidate["column"]] > best_candidate["threshold"]]
print(f"prices {right_prices.tolist()}: squared error {((right_prices - right_prices.mean()) ** 2).mean():.3f},"
      f" absolute error {(right_prices - right_prices.median()).abs().mean():.3f}")

**The right child of the toy split holds the prices 14, 17, 9, 12 and 8: its squared-error impurity is 10.800 and its absolute error around the median is 2.800.** The two criteria measure one group in squared thousand euros and in thousand euros.

__Step 29:__ Set the criteria to try: `["squared_error", "absolute_error", "poisson"]`. `"squared_error"` is the default.

In [ ]:
criterion_values = ["squared_error", "absolute_error", "poisson"]

In [ ]:
criterion_rows = []
for value in tqdm(criterion_values, desc='criterion values', leave=True, position=0):
    model = DecisionTreeRegressor(criterion=value, random_state=random_state)
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in tree_prepared]
    fitted = clone(model).fit(Z_train, y_train)
    row = {"configuration": f"DecisionTreeRegressor(criterion={value!r})", "setting": str(value),
           "train": [mean_absolute_error(fold["y_fit"], tree.predict(fold["X_fit"]))
                     for fold, tree in zip(tree_prepared, fold_models)],
           "validation": [mean_absolute_error(fold["y_score"], tree.predict(fold["X_score"]))
                          for fold, tree in zip(tree_prepared, fold_models)],
           "model": fitted}
    row["train_MAE"], row["cv_MAE"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_MAE"] = mean_absolute_error(y_test, fitted.predict(Z_test))
    row["depth"], row["leaves"] = fitted.get_depth(), fitted.get_n_leaves()
    criterion_rows.append(row)
criterion_sweep = pd.DataFrame(criterion_rows)
criterion_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "depth", "leaves"]].round(2)

In [ ]:
# The sweep's winner joins the log.
best = criterion_sweep.loc[criterion_sweep["cv_MAE"].idxmin()]
record(tree_log, {"frame": type(tree_folds).__name__, "rows": len(y_train),
                  "folds": tree_folds.get_n_splits(), "name": best["configuration"],
                  "train": best["train"], "validation": best["validation"], "model": best["model"]},
       Z_test, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

In [ ]:
SuperTree(best['model'], Z_train, y_train.to_numpy(), prepared_names, None).show_tree(start_depth=best['model'].get_depth(), max_samples=500)

**`'absolute_error'` does not lower validation MAE: it scores 3,068.14 EUR against 3,053.52 for `'squared_error'`, so the prediction does not hold, although its test MAE is lower, 2,723.89 against 2,831.21; `'poisson'` does lower it, to 3,008.43, with a test MAE of 2,848.32.** Their trees have 3,156 and 3,162 leaves, against 3,161. Validation selects, so `DecisionTreeRegressor(criterion='poisson')` joins the log.

**`max_features`: how many columns a split may consider.** `None` offers every prepared column to every split. `'sqrt'` and `'log2'` offer a random subset of that many columns, drawn again at every node; if none of them gives a valid split, the search continues through the others. A tree that sees fewer columns at each node cannot always take the best one.

**Prediction:** restricting the columns raises validation MAE for a single tree.

__Step 30:__ Set the column counts to try: `[None, "sqrt", "log2"]`. `None` is the default.

In [ ]:
max_features_values = [None, "sqrt", "log2"]

In [ ]:
max_features_rows = []
for value in tqdm(max_features_values, desc='max features values', leave=True, position=0):
    model = DecisionTreeRegressor(max_features=value, random_state=random_state)
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in tree_prepared]
    fitted = clone(model).fit(Z_train, y_train)
    row = {"configuration": f"DecisionTreeRegressor(max_features={value!r})", "setting": str(value),
           "train": [mean_absolute_error(fold["y_fit"], tree.predict(fold["X_fit"]))
                     for fold, tree in zip(tree_prepared, fold_models)],
           "validation": [mean_absolute_error(fold["y_score"], tree.predict(fold["X_score"]))
                          for fold, tree in zip(tree_prepared, fold_models)],
           "model": fitted}
    row["train_MAE"], row["cv_MAE"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_MAE"] = mean_absolute_error(y_test, fitted.predict(Z_test))
    row["depth"], row["leaves"] = fitted.get_depth(), fitted.get_n_leaves()
    max_features_rows.append(row)
max_features_sweep = pd.DataFrame(max_features_rows)
max_features_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "depth", "leaves"]].round(2)

In [ ]:
# The sweep's winner joins the log.
best = max_features_sweep.loc[max_features_sweep["cv_MAE"].idxmin()]
record(tree_log, {"frame": type(tree_folds).__name__, "rows": len(y_train),
                  "folds": tree_folds.get_n_splits(), "name": best["configuration"],
                  "train": best["train"], "validation": best["validation"], "model": best["model"]},
       Z_test, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

**`'sqrt'` and `'log2'` give identical results, a validation MAE of 3,478.32 EUR against 3,053.52 for `None`, because both offer 4 of the 21 columns at each split.** The trees keep 3,180 leaves. The default wins, so `DecisionTreeRegressor(max_features=None)` joins the log with the default's scores, and its tree is the default tree drawn above.

**`ccp_alpha`: the price of a leaf.** After the tree is grown, cost-complexity pruning removes every branch whose impurity saving is less than `ccp_alpha` for each leaf it adds; the formula is in the pruning section below. The saving is measured in the units of the impurity, here squared euros, so a useful value is far from the values that prune a Gini tree.

**Prediction:** a larger `ccp_alpha` gives a smaller tree and a higher training MAE, and validation MAE falls until the pruning removes branches that were useful.

__Step 31:__ Set the values of `ccp_alpha` to try: `[0.0, 10000.0, 100000.0, 1000000.0]`. At `0.0` it is at its default.

In [ ]:
ccp_alpha_values = [0.0, 10000.0, 100000.0, 1000000.0]

In [ ]:
ccp_alpha_rows = []
for value in tqdm(ccp_alpha_values, desc='ccp alpha values', leave=True, position=0):
    model = DecisionTreeRegressor(ccp_alpha=value, random_state=random_state)
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in tree_prepared]
    fitted = clone(model).fit(Z_train, y_train)
    row = {"configuration": f"DecisionTreeRegressor(ccp_alpha={value!r})", "setting": str(value),
           "train": [mean_absolute_error(fold["y_fit"], tree.predict(fold["X_fit"]))
                     for fold, tree in zip(tree_prepared, fold_models)],
           "validation": [mean_absolute_error(fold["y_score"], tree.predict(fold["X_score"]))
                          for fold, tree in zip(tree_prepared, fold_models)],
           "model": fitted}
    row["train_MAE"], row["cv_MAE"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_MAE"] = mean_absolute_error(y_test, fitted.predict(Z_test))
    row["depth"], row["leaves"] = fitted.get_depth(), fitted.get_n_leaves()
    ccp_alpha_rows.append(row)
ccp_alpha_sweep = pd.DataFrame(ccp_alpha_rows)
ccp_alpha_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "depth", "leaves"]].round(2)

In [ ]:
# The sweep's winner joins the log.
best = ccp_alpha_sweep.loc[ccp_alpha_sweep["cv_MAE"].idxmin()]
record(tree_log, {"frame": type(tree_folds).__name__, "rows": len(y_train),
                  "folds": tree_folds.get_n_splits(), "name": best["configuration"],
                  "train": best["train"], "validation": best["validation"], "model": best["model"]},
       Z_test, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

In [ ]:
SuperTree(best['model'], Z_train, y_train.to_numpy(), prepared_names, None).show_tree(start_depth=best['model'].get_depth(), max_samples=500)

**`ccp_alpha=10000.0` gives the lowest validation MAE of its sweep, 2,921.17 EUR with 350 leaves, against 3,053.52 at 0.0, while 100,000.0 and 1,000,000.0 prune to 62 and 15 leaves and score 3,140.46 and 3,646.15.** The values are in squared euros. `DecisionTreeRegressor(ccp_alpha=10000.0)` joins the log.

__Step 32:__ Set the values of `ccp_alpha` for `criterion="absolute_error"`, whose impurity is in euros: `[0.0, 1.0, 10.0, 100.0]`.

In [ ]:
ccp_alpha_absolute_values = [0.0, 1.0, 10.0, 100.0]

In [ ]:
ccp_alpha_absolute_rows = []
for value in tqdm(ccp_alpha_absolute_values, desc='ccp alpha absolute values', leave=True, position=0):
    model = DecisionTreeRegressor(criterion="absolute_error", ccp_alpha=value, random_state=random_state)
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in tree_prepared]
    fitted = clone(model).fit(Z_train, y_train)
    row = {"configuration": f"DecisionTreeRegressor(criterion='absolute_error', ccp_alpha={value!r})", "setting": str(value),
           "train": [mean_absolute_error(fold["y_fit"], tree.predict(fold["X_fit"]))
                     for fold, tree in zip(tree_prepared, fold_models)],
           "validation": [mean_absolute_error(fold["y_score"], tree.predict(fold["X_score"]))
                          for fold, tree in zip(tree_prepared, fold_models)],
           "model": fitted}
    row["train_MAE"], row["cv_MAE"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_MAE"] = mean_absolute_error(y_test, fitted.predict(Z_test))
    row["depth"], row["leaves"] = fitted.get_depth(), fitted.get_n_leaves()
    ccp_alpha_absolute_rows.append(row)
ccp_alpha_absolute_sweep = pd.DataFrame(ccp_alpha_absolute_rows)
ccp_alpha_absolute_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "depth", "leaves"]].round(2)

In [ ]:
# The sweep's winner joins the log.
best = ccp_alpha_absolute_sweep.loc[ccp_alpha_absolute_sweep["cv_MAE"].idxmin()]
record(tree_log, {"frame": type(tree_folds).__name__, "rows": len(y_train),
                  "folds": tree_folds.get_n_splits(), "name": best["configuration"],
                  "train": best["train"], "validation": best["validation"], "model": best["model"]},
       Z_test, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

In [ ]:
SuperTree(best['model'], Z_train, y_train.to_numpy(), prepared_names, None).show_tree(start_depth=best['model'].get_depth(), max_samples=500)

**With `criterion='absolute_error'`, whose impurity is in euros, `ccp_alpha=1.0` is best, at a validation MAE of 2,999.44 EUR against 3,068.14 at 0.0, and 100.0 prunes to 10 leaves and 3,641.93.** The useful value is ten thousand times smaller than the squared-error sweep's, because this impurity is in euros and not in squared euros. `DecisionTreeRegressor(criterion='absolute_error', ccp_alpha=1.0)` joins the log.

__Step 33:__ Set the values of `ccp_alpha` for `criterion="poisson"`, whose impurity is half the Poisson deviance of the prices: `[0.0, 0.1, 1.0, 10.0]`.

In [ ]:
ccp_alpha_poisson_values = [0.0, 0.1, 1.0, 10.0]

In [ ]:
ccp_alpha_poisson_rows = []
for value in tqdm(ccp_alpha_poisson_values, desc='ccp alpha poisson values', leave=True, position=0):
    model = DecisionTreeRegressor(criterion="poisson", ccp_alpha=value, random_state=random_state)
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in tree_prepared]
    fitted = clone(model).fit(Z_train, y_train)
    row = {"configuration": f"DecisionTreeRegressor(criterion='poisson', ccp_alpha={value!r})", "setting": str(value),
           "train": [mean_absolute_error(fold["y_fit"], tree.predict(fold["X_fit"]))
                     for fold, tree in zip(tree_prepared, fold_models)],
           "validation": [mean_absolute_error(fold["y_score"], tree.predict(fold["X_score"]))
                          for fold, tree in zip(tree_prepared, fold_models)],
           "model": fitted}
    row["train_MAE"], row["cv_MAE"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_MAE"] = mean_absolute_error(y_test, fitted.predict(Z_test))
    row["depth"], row["leaves"] = fitted.get_depth(), fitted.get_n_leaves()
    ccp_alpha_poisson_rows.append(row)
ccp_alpha_poisson_sweep = pd.DataFrame(ccp_alpha_poisson_rows)
ccp_alpha_poisson_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "depth", "leaves"]].round(2)

In [ ]:
# The sweep's winner joins the log.
best = ccp_alpha_poisson_sweep.loc[ccp_alpha_poisson_sweep["cv_MAE"].idxmin()]
record(tree_log, {"frame": type(tree_folds).__name__, "rows": len(y_train),
                  "folds": tree_folds.get_n_splits(), "name": best["configuration"],
                  "train": best["train"], "validation": best["validation"], "model": best["model"]},
       Z_test, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

In [ ]:
SuperTree(best['model'], Z_train, y_train.to_numpy(), prepared_names, None).show_tree(start_depth=best['model'].get_depth(), max_samples=500)

**With `criterion='poisson'`, `ccp_alpha=1.0` is best, at a validation MAE of 2,891.79 EUR with 130 leaves against 3,008.43 at 0.0, and 10.0 prunes to 20 leaves and 3,388.63.** The impurity is half a deviance, so the useful value is far from those of the squared-error and absolute-error sweeps. `DecisionTreeRegressor(criterion='poisson', ccp_alpha=1.0)` joins the log.

__Step 34:__ Set the values of `ccp_alpha` for a tree fitted to `log(price)`, whose impurity is in squared log euros: `[0.0, 0.0001, 0.001, 0.01]`.

In [ ]:
ccp_alpha_log_values = [0.0, 0.0001, 0.001, 0.01]

In [ ]:
ccp_alpha_log_rows = []
for value in tqdm(ccp_alpha_log_values, desc='ccp alpha log values', leave=True, position=0):
    model = TransformedTargetRegressor(regressor=DecisionTreeRegressor(ccp_alpha=value, random_state=random_state), func=np.log, inverse_func=np.exp)
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in tree_prepared]
    fitted = clone(model).fit(Z_train, y_train)
    row = {"configuration": f"DecisionTreeRegressor(ccp_alpha={value!r}), log(price)", "setting": str(value),
           "train": [mean_absolute_error(fold["y_fit"], tree.predict(fold["X_fit"]))
                     for fold, tree in zip(tree_prepared, fold_models)],
           "validation": [mean_absolute_error(fold["y_score"], tree.predict(fold["X_score"]))
                          for fold, tree in zip(tree_prepared, fold_models)],
           "model": fitted}
    row["train_MAE"], row["cv_MAE"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_MAE"] = mean_absolute_error(y_test, fitted.predict(Z_test))
    row["depth"], row["leaves"] = fitted.regressor_.get_depth(), fitted.regressor_.get_n_leaves()
    ccp_alpha_log_rows.append(row)
ccp_alpha_log_sweep = pd.DataFrame(ccp_alpha_log_rows)
ccp_alpha_log_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "depth", "leaves"]].round(2)

In [ ]:
# The sweep's winner joins the log.
best = ccp_alpha_log_sweep.loc[ccp_alpha_log_sweep["cv_MAE"].idxmin()]
record(tree_log, {"frame": type(tree_folds).__name__, "rows": len(y_train),
                  "folds": tree_folds.get_n_splits(), "name": best["configuration"],
                  "train": best["train"], "validation": best["validation"], "model": best["model"]},
       Z_test, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

In [ ]:
SuperTree(best['model'].regressor_, Z_train, np.log(y_train.to_numpy()), prepared_names, None).show_tree(start_depth=best['model'].regressor_.get_depth(), max_samples=500)

**On `log(price)`, whose impurity is in squared log euros, `ccp_alpha=0.0001` is best, at a validation MAE of 2,780.49 EUR with 153 leaves against 3,008.90 at 0.0, and 0.001 prunes to 20 leaves and 3,261.03.** `DecisionTreeRegressor(ccp_alpha=0.0001), log(price)` joins the log.

**The target scale: `log(price)`.** Fitting the tree to the logarithm of the price changes the numbers whose squared deviations the impurity adds up, so it can change which splits win, unlike a change of units in a feature. Each leaf then holds the mean of the log prices of its cars, and exponentiating it gives their geometric mean, which is below their arithmetic mean. MAE is still measured in euros on the original prices. In a drawn box of such a tree, `value` is in log units, and its exponential is a price.

**Prediction:** the log scale changes the trees, and validation MAE moves with it.

__Step 35:__ Set the target scales to try: `["price", "log(price)"]`. `"price"` is the default.

In [ ]:
target_values = ["price", "log(price)"]

In [ ]:
target_rows = []
for value in tqdm(target_values, desc='target values', leave=True, position=0):
    model = DecisionTreeRegressor(random_state=random_state)
    if value == "log(price)":
        model = TransformedTargetRegressor(regressor=model, func=np.log, inverse_func=np.exp)
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in tree_prepared]
    fitted = clone(model).fit(Z_train, y_train)
    tree = fitted.regressor_ if value == "log(price)" else fitted
    row = {"configuration": f"DecisionTreeRegressor(), {value}", "setting": value,
           "train": [mean_absolute_error(fold["y_fit"], tree_fold.predict(fold["X_fit"]))
                     for fold, tree_fold in zip(tree_prepared, fold_models)],
           "validation": [mean_absolute_error(fold["y_score"], tree_fold.predict(fold["X_score"]))
                          for fold, tree_fold in zip(tree_prepared, fold_models)],
           "model": fitted}
    row["train_MAE"], row["cv_MAE"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_MAE"] = mean_absolute_error(y_test, fitted.predict(Z_test))
    row["depth"], row["leaves"] = tree.get_depth(), tree.get_n_leaves()
    target_rows.append(row)
target_sweep = pd.DataFrame(target_rows)
target_sweep.set_index("setting")[["train_MAE", "cv_MAE", "test_MAE", "depth", "leaves"]].round(2)

In [ ]:
# The sweep's winner joins the log.
best = target_sweep.loc[target_sweep["cv_MAE"].idxmin()]
record(tree_log, {"frame": type(tree_folds).__name__, "rows": len(y_train),
                  "folds": tree_folds.get_n_splits(), "name": best["configuration"],
                  "train": best["train"], "validation": best["validation"], "model": best["model"]},
       Z_test, y_test, "one setting changed", metric=mean_absolute_error, label="MAE")

In [ ]:
SuperTree(best['model'].regressor_, Z_train, np.log(y_train.to_numpy()), prepared_names, None).show_tree(start_depth=best['model'].regressor_.get_depth(), max_samples=500)

**Fitting to `log(price)` lowers validation MAE from 3,053.52 EUR to 3,008.90, with a similar tree, 3,175 leaves against 3,161, and a test MAE of 2,818.29.** The log scale changes the trees and the score, as predicted, and `DecisionTreeRegressor(), log(price)` joins the log.

# <font color='#E8800A'>Two ways to prune</font> <a class="anchor" id="prune"></a>
[Back to TOC](#toc)

**Pre-pruning** stops the tree before it grows: `max_depth` limits a path, `min_samples_split` limits the cars a node needs before it may split, and `min_samples_leaf` limits the cars a child may hold. **Post-pruning** grows the tree first and then removes branches. Cost-complexity pruning keeps, for a given $\alpha$, the subtree $T$ that minimises

$$R_\alpha(T) = R(T) + \alpha\,\lvert T \rvert,$$

where $R(T)$ is the sum over the leaves of $T$ of each leaf's squared-error impurity weighted by its share of the training cars, $\lvert T \rvert$ is the number of leaves, and $\alpha$ is `ccp_alpha`, the price of one leaf. A branch is removed when the impurity it saves is less than $\alpha$ for each leaf it adds. Impurity is measured in squared euros, so $\alpha$ is too, and the path lists the values of $\alpha$ at which branches disappear, computed from the training rows alone.

__Step 36:__ Compute the pruning path of the default tree on the training cars: the alphas at which branches would be removed.

In [ ]:
pruning_path = default_fitted.cost_complexity_pruning_path(Z_train, y_train)
path_alphas = pruning_path.ccp_alphas[:-1]
print(f"{len(path_alphas)} effective alphas, from {path_alphas.min():.0f} to {path_alphas.max():.0f}, and"
      f" {pruning_path.ccp_alphas[-1]:.0f} for the last, which leaves the root alone")

__Step 37:__ Fit a pruned tree at 40 of those alphas, and count the leaves of each.

In [ ]:
alphas = np.unique(np.quantile(path_alphas, np.linspace(0, 1, 40)))
pruned_leaves = [DecisionTreeRegressor(ccp_alpha=alpha, random_state=random_state).fit(Z_train, y_train).get_n_leaves()
                 for alpha in tqdm(alphas, desc='alphas', leave=True, position=0)]

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(alphas[1:], pruned_leaves[1:], marker="o", markersize=5, linewidth=2, color=PLOT_BLUE)
for value in ccp_alpha_values[1:]:
    plt.axvline(value, color="grey", linestyle=":", linewidth=1.5)
plt.xscale("log")
plt.xlabel("ccp_alpha (log scale); dotted: the values swept above")
plt.ylabel("leaves of the pruned tree")
plt.title("Leaves left after cost-complexity pruning")
plt.show()

**The pruned tree keeps fewer leaves as `ccp_alpha` rises, and most of its leaves go between 1 and 10,000.** At the dotted lines the sweep left 350 leaves at 10,000, 62 at 100,000 and 15 at 1,000,000.

__Step 38:__ The winner of each pruning sweep, beside the default: its depth, leaves, and validation and test scores.

In [ ]:
pruning_rows = [{"kind": "none", "tree": "DecisionTreeRegressor()", "depth": default_fitted.get_depth(),
                 "leaves": default_fitted.get_n_leaves(), "cv_MAE": np.mean(default_cv["validation"]),
                 "test_MAE": mean_absolute_error(y_test, default_fitted.predict(Z_test))}]
for kind, sweep in [("pre-pruning", max_depth_sweep), ("pre-pruning", min_samples_leaf_sweep),
                    ("pre-pruning", min_samples_split_sweep), ("post-pruning", ccp_alpha_sweep)]:
    best = sweep.loc[sweep["cv_MAE"].idxmin()]
    pruning_rows.append({"kind": kind, "tree": best["configuration"], "depth": best["depth"],
                         "leaves": best["leaves"], "cv_MAE": best["cv_MAE"], "test_MAE": best["test_MAE"]})
pd.DataFrame(pruning_rows).set_index("tree").round(2)

__Step 39:__ Print the mean training and validation MAE of the `ccp_alpha` sweep, and the gap between them.

In [ ]:
pd.DataFrame({"ccp_alpha": ccp_alpha_values, "training": ccp_alpha_sweep["train_MAE"],
              "validation": ccp_alpha_sweep["cv_MAE"],
              "gap": ccp_alpha_sweep["cv_MAE"] - ccp_alpha_sweep["train_MAE"]}).set_index("ccp_alpha").round(2)

__Step 40:__ Plot the training and validation MAE of the `ccp_alpha` sweep against `ccp_alpha`, with the chosen value marked.

In [ ]:
chosen_alpha = ccp_alpha_sweep["cv_MAE"].idxmin()
plt.figure(figsize=(7, 5))
plt.plot(ccp_alpha_values, ccp_alpha_sweep["train_MAE"], marker="o", markersize=8, linewidth=2,
         color=PLOT_ORANGE, label="training (ten folds)", clip_on=False)
plt.plot(ccp_alpha_values, ccp_alpha_sweep["cv_MAE"], marker="o", markersize=8, linewidth=2,
         color=PLOT_BLUE, label="validation (ten folds)", clip_on=False)
plt.axvline(ccp_alpha_values[chosen_alpha], color="grey", linestyle=":", linewidth=1.5, label="chosen ccp_alpha")
plt.xscale("symlog", linthresh=ccp_alpha_values[1])
plt.xlim(0, ccp_alpha_values[-1] * 1.5)
plt.xlabel("ccp_alpha (symlog scale)")
plt.ylabel("MAE")
plt.title("Training and validation MAE against ccp_alpha")
plt.legend()
plt.show()

**Training MAE rises from 0.66 EUR to 3,392.24 as `ccp_alpha` rises to 1,000,000.0, while validation MAE falls from 3,053.52 to 2,921.17 at 10,000.0 and rises to 3,646.15.** The two lines start far apart and end close together, and the dotted line marks the value the sweep chose.

__Step 41:__ Draw the post-pruned tree of the `ccp_alpha` sweep, every node; the default tree is drawn above.

In [ ]:
post_pruned = ccp_alpha_sweep.loc[ccp_alpha_sweep["cv_MAE"].idxmin(), "model"]

In [ ]:
SuperTree(post_pruned, Z_train, y_train.to_numpy(), prepared_names, None).show_tree(start_depth=post_pruned.get_depth(), max_samples=500)

**Pre-pruning and post-pruning reach similar validation MAE with trees of different sizes:** `max_depth=12` gives 2,925.67 EUR with 1,008 leaves, `min_samples_leaf=5` 2,795.38 with 521, `min_samples_split=10` 2,838.23 with 720 and `ccp_alpha=10000.0` 2,921.17 with 350, against 3,053.52 for the default's 3,161. The post-pruned tree has depth 21 and 350 leaves, so pruning removed branches of the deep tree without capping its depth.

<div class="alert alert-block alert-warning">

**The scores above describe these folds.** A setting that scores higher on the validation folds has not been shown better in general: the ten folds share cars, so a gap between two settings is a descriptive difference and not a significance test, and following a good reason for a setting does not guarantee a better score. Here `splitter` and `max_features` ended on the default, while `criterion='poisson'` and limiting growth lowered validation MAE.

</div>

# <font color='#E8800A'>Nested cross-validation</font> <a class="anchor" id="nested"></a>
[Back to TOC](#toc)

The grid fits every combination of the settings: 320, in four groups of 64, 64, 64 and 128, because impurity is in different units. Trees on the price with `criterion='squared_error'`, `criterion='absolute_error'` and `criterion='poisson'` take their own `ccp_alpha` values from their own sweeps, and trees on `log(price)` take a fourth list. Each setting takes its default and its best other value from its sweep. Nested cross-validation estimates how that whole grid procedure scores on cars that took no part in its choice: the grid runs inside each of the ten outer folds on the outer training part, and the outer validation part scores its winner. It returns no model, and its mean is read beside the default's validation MAE on the same folds.

__Step 42:__ Define the grid in four groups, one for each impurity. Each setting takes its default and its best other value from its own sweep. Group 1, `criterion="squared_error"`: `max_depth: [12, None], min_samples_leaf: [1, 5], min_samples_split: [2, 10], splitter: ["best", "random"], max_features: [None, "sqrt"], ccp_alpha: [0.0, 10000.0]`. Type each list of group 1 as written. Group 2, `criterion="absolute_error"`, repeats those lists with `ccp_alpha: [0.0, 1.0]`. Group 3, `criterion="poisson"`, repeats them with `ccp_alpha: [0.0, 1.0]`. Group 4, on `log(price)`, repeats them with `criterion: ["squared_error", "absolute_error"]` and `ccp_alpha: [0.0, 0.0001]`. Groups 2, 3 and 4 are given.

In [ ]:
tree_grid = [
    {"estimator": [DecisionTreeRegressor(random_state=random_state)],
     "estimator__max_depth": ..., "estimator__min_samples_leaf": ...,
     "estimator__min_samples_split": ..., "estimator__criterion": ["squared_error"],
     "estimator__splitter": ...,
     "estimator__max_features": ..., "estimator__ccp_alpha": ...},
    {"estimator": [DecisionTreeRegressor(random_state=random_state)],
     "estimator__max_depth": [12, None], "estimator__min_samples_leaf": [1, 5],
     "estimator__min_samples_split": [2, 10], "estimator__criterion": ["absolute_error"],
     "estimator__splitter": ["best", "random"],
     "estimator__max_features": [None, "sqrt"], "estimator__ccp_alpha": [0.0, 1.0]},
    {"estimator": [DecisionTreeRegressor(random_state=random_state)],
     "estimator__max_depth": [12, None], "estimator__min_samples_leaf": [1, 5],
     "estimator__min_samples_split": [2, 10], "estimator__criterion": ["poisson"],
     "estimator__splitter": ["best", "random"],
     "estimator__max_features": [None, "sqrt"], "estimator__ccp_alpha": [0.0, 1.0]},
    {"estimator": [TransformedTargetRegressor(
         regressor=DecisionTreeRegressor(random_state=random_state), func=np.log, inverse_func=np.exp)],
     "estimator__regressor__max_depth": [12, None], "estimator__regressor__min_samples_leaf": [1, 5],
     "estimator__regressor__min_samples_split": [2, 10],
     "estimator__regressor__criterion": ["squared_error", "absolute_error"],
     "estimator__regressor__splitter": ["best", "random"],
     "estimator__regressor__max_features": [None, "sqrt"],
     "estimator__regressor__ccp_alpha": [0.0, 0.0001]}]
tree_template = CachedSearchCV(
    PreparedEstimator(FoldPreprocessor(regression_recipe), DecisionTreeRegressor(random_state=random_state)),
    tree_grid, scoring="neg_mean_absolute_error", cv=tree_folds, n_jobs=4,
)
print(f"{len(ParameterGrid(tree_grid))} combinations, each scored on {tree_folds.get_n_splits()} inner folds")

**320 combinations on ten inner folds is 3,200 fits for one grid, so nested cross-validation fits 32,000 candidates and ten winners, and the final grid 3,200 and one.** Candidates fitted on the same rows share their fold preprocessing, and four workers fit the folds in parallel. Absolute-error trees take longer to fit than squared-error trees, so the count bounds the work and does not predict its duration.

__Step 43:__ Estimate the grid with nested cross-validation.

In [ ]:
tree_nested_fold_results = [cross_validate(tree_template, X_train, y_train, cv=[_outer_split], scoring='neg_mean_absolute_error', return_train_score=True)
    for _outer_split in tqdm(list(tree_folds.split(X_train, y_train)), total=tree_folds.get_n_splits(X_train, y_train), desc='Nested CV: outer folds', unit='fold', leave=True, position=0)]
tree_nested = {key: np.concatenate([np.asarray(result[key], dtype=object if key == 'estimator' else None) for result in tree_nested_fold_results]) for key in tree_nested_fold_results[0]}
nested_validation = -tree_nested["test_score"]
print(f"outer training MAE {-tree_nested['train_score'].mean():,.2f} EUR; "
      f"outer validation MAE {nested_validation.mean():,.2f} EUR, fold SD {nested_validation.std(ddof=1):,.2f}")

**Nested validation MAE is 2,753.22 EUR over the ten outer folds, with a fold standard deviation of 171.78.** The outer training MAE, 2,047.93, is 705.29 below it, against the 3,052.86 of the default. The folds share cars, so 171.78 is the spread of ten overlapping scores and not a standard error.

__Step 44:__ Set the default's validation score on the same folds beside the nested estimate, and compare their paired fold scores.

In [ ]:
paired_gain = np.array(default_cv["validation"]) - nested_validation
print(f"the default on the same folds: MAE {np.mean(default_cv['validation']):,.2f} EUR")
print(f"default minus nested: mean {paired_gain.mean():+,.2f};"
      f" fold range {paired_gain.min():+,.2f} to {paired_gain.max():+,.2f};"
      f" grid lower in {(paired_gain > 0).sum()} of {len(paired_gain)} folds")

**The nested estimate is 300.30 EUR below the default's 3,053.52, and the grid is lower in all ten folds, by 7.65 to 526.64.** This is a descriptive comparison of overlapping splits, not a significance test.

# <font color='#E8800A'>The grid search</font> <a class="anchor" id="search"></a>
[Back to TOC](#toc)

The grid now runs once on all the training cars, and its winner is the model this week deploys.

__Step 45:__ Run the grid on all the training cars.

In [ ]:
tree_search = clone(tree_template).fit(X_train, y_train)

__Step 46:__ List the five best combinations, each beside the test score of its refit.

In [ ]:
results = tree_search.cv_results_
ranking = pd.DataFrame({
    "target": ["log(price)" if isinstance(candidate["estimator"], TransformedTargetRegressor) else "price"
               for candidate in results["params"]],
    "settings": [", ".join(f"{name.split('__')[-1]}={value}" for name, value in candidate.items()
                           if name != "estimator") for candidate in results["params"]],
    "cv_MAE": -results["mean_test_score"],
}).nsmallest(5, "cv_MAE")
# Each candidate is refitted on the training rows, and only then scored on the test rows.
ranking["test_MAE"] = ranking.index.map(lambda index: mean_absolute_error(
    y_test, clone(tree_search.estimator).set_params(
        **clone(results["params"][index], safe=False)).fit(X_train, y_train).predict(X_test)))
print(f"{len(results['params'])} combinations; the five best:")
print(ranking.round(2).to_string())

**All five best combinations fit `log(price)` with `criterion='absolute_error'`, `max_features=None` and `min_samples_leaf=5`, at validation MAE 2,710.20 to 2,734.21 EUR.** The first two, `min_samples_split=2` and `10`, are the same tree, since a leaf of at least 5 cars needs a node of at least 10, so they tie and the first listed wins, with a test MAE of 2,661.94.

__Step 47:__ Log the winner's own score: its validation score on the same folds.

In [ ]:
# The winner's own score is the best of all of them on the same folds: its settings
# are fitted on each fold's prepared training part.
chosen_tree = tree_search.best_estimator_
fold_models = [clone(chosen_tree.estimator).fit(fold["X_fit"], fold["y_fit"]) for fold in tqdm(tree_prepared, desc='tree prepared', leave=True, position=0)]
winner_cv = {"train": [mean_absolute_error(fold["y_fit"], tree.predict(fold["X_fit"]))
                       for fold, tree in zip(tree_prepared, fold_models)],
             "validation": [mean_absolute_error(fold["y_score"], tree.predict(fold["X_score"]))
                            for fold, tree in zip(tree_prepared, fold_models)]}
# The log names the winner by its settings.
winner_settings = ranking.loc[tree_search.best_index_]
tree_winner_name = (f"DecisionTreeRegressor({winner_settings['settings']}), "
                    f"{winner_settings['target']}")
record(tree_log, {"frame": type(tree_folds).__name__, "rows": len(y_train),
                  "folds": tree_folds.get_n_splits(), "name": tree_winner_name,
                  "train": winner_cv["train"], "validation": winner_cv["validation"],
                  "model": chosen_tree},
       X_test, y_test, "winner's own score", metric=mean_absolute_error, label="MAE")

__Step 48:__ Log the winner's nested estimate, beside the same deployed model's test score.

In [ ]:
record(tree_log, {"frame": type(tree_folds).__name__, "rows": len(y_train),
                  "folds": tree_folds.get_n_splits(),
                  "name": f"joint search over {len(tree_search.cv_results_['params'])} combinations",
                  "train": -tree_nested["train_score"], "validation": nested_validation,
                  "model": chosen_tree},
       X_test, y_test, "nested estimate", metric=mean_absolute_error, label="MAE")

**The winner's own validation MAE, 2,710.20 EUR, is below its nested estimate, 2,753.22,** because it was chosen as the best of 320 on the same folds. The board publishes the nested estimate.

__Step 49:__ Draw the grid winner's tree, every node, in the zoomable viewer.

In [ ]:
winner_tree = getattr(chosen_tree.estimator_, "regressor_", chosen_tree.estimator_)

In [ ]:
SuperTree(winner_tree, Z_train, np.log(y_train.to_numpy()) if hasattr(chosen_tree.estimator_, "regressor_") else y_train.to_numpy(), prepared_names, None).show_tree(start_depth=winner_tree.get_depth(), max_samples=500)

**The grid winner has 16 levels and 270 leaves, against the default's 28 and 3,161, and its top question is the default's, about `transmission_Manual`.** It fits `log(price)`, so its leaves state log prices.

# <font color='#E8800A'>The number we stand behind</font> <a class="anchor" id="board"></a>
[Back to TOC](#toc)

The grid's winner was chosen on the training folds. The closing comparison reports its test score beside the default, and the board publishes that score alongside the nested estimate.

__Step 50:__ Report the winner and the default on the test cars.

In [ ]:
tuned_mae = mean_absolute_error(y_test, chosen_tree.predict(X_test))
default_mae = mean_absolute_error(y_test, default_fitted.predict(Z_test))
print(f"winner test MAE: {tuned_mae:,.2f} EUR")
print(f"default test MAE: {default_mae:,.2f} EUR")

**On the 800 held-out cars the winner scores MAE 2,661.94 EUR and the default 2,831.21,** against 2,753.22 nested and 3,053.52 on the folds. The configuration was chosen with validation scores alone; the test scores printed during the sweeps did not select it.

__Step 51:__ Take the default's cross-validated score, on the same folds, as the baseline.

In [ ]:
# The default was measured on these same folds, so it is the baseline.
default_cv_mean = np.mean(default_cv["validation"])
print(f"default over the outer folds: MAE {default_cv_mean:,.2f} EUR")

__Step 52:__ Build the course scoreboard with the results earlier weeks saved and the floor.

In [ ]:
# The board: a result an earlier week saved joins it only if it was scored on the
# same data, recipe, rows and folds; the floor predicts every car at the training mean.
results_dir = "../../results"
tree_board = Scoreboard("cars4you resale", "MAE (EUR)", context=result_context(
    "../../data/interim/cars4you.csv", regression_recipe_log,
    y_train.index.tolist(), y_test.index.tolist(),
    outer_folds=outer_folds(tree_folds, X_train, y_train)),
    floor=round(mean_absolute_error(y_test, DummyRegressor(strategy="mean").fit(
        X_train, y_train).predict(X_test)), 4))
for week, model, cv, test in results_before(9, "regression", results_dir,
                                             metric=tree_board.metric,
                                             context=tree_board.context):
    tree_board.record(week, model, cv, test)

__Step 53:__ Record this week's result on the board, save it with the fitted winner, and print the board.

In [ ]:
# The row carries two numbers: the nested cross-validated mean first, then the
# winner's held-out score. Its label names every setting the grid chose, from best_params_.
settings = tree_search.best_params_
chosen_label = "DecisionTreeRegressor(" + ", ".join(
    f"{name.split('__')[-1]}={value!r}"
    for name, value in settings.items() if name != "estimator") + ")"
if isinstance(settings["estimator"], TransformedTargetRegressor):
    chosen_label += ", log(price)"
tree_board.record(9, chosen_label, round(nested_validation.mean(), 4), round(tuned_mae, 4))
tree_board.save(
    results_dir, 9, "regression", baseline_cv=round(default_cv_mean, 4),
    baseline_test=round(default_mae, 4),
)

# The fitted winner is saved beside its result, so a later week can reload it if
# it ends the course as the best model. Loading needs preprocessing.py on the path.
joblib.dump(chosen_tree, f"{results_dir}/week_09_regression_model.joblib")

print(tree_board.table())

<div class="alert alert-block alert-success">

**A tree on `log(price)` with `criterion='absolute_error'`, `ccp_alpha=0.0001`, `min_samples_leaf=5` and no depth limit joins the board at MAE 2,753.22 EUR over the ten outer folds and 2,661.94 held out,** against a floor of 6,843.46. The Lasso on log(price) leads on the folds, at 2,546.51, and KNN on the held-out cars, at 2,564.17, so the columns order those models differently. The row and the fitted winner are saved in `results/`.

</div>

# <font color='#E8800A'>Key takeaways</font> <a class="anchor" id="takeaways"></a>
[Back to TOC](#toc)

1. **A split is chosen by the impurity reduction of its children:** on the toy cars the best question, `age <= 3.5`, lowers the weighted squared error from 35.359 to 7.000.
2. **A prediction is the mean of one leaf:** the first test car is priced at 10,560.09 EUR from 1,353 training cars, and the unrestricted tree's leaves hold a median of 1 car.
3. **Limiting growth beat the default on validation:** MAE 3,053.52 EUR unrestricted, against 2,925.67 at `max_depth=12`, 2,795.38 at `min_samples_leaf=5` and 2,921.17 at `ccp_alpha=10000.0`.
4. **Nested cross-validation put the grid at 2,753.22 EUR, below the default's 3,053.52 in all ten folds,** and its winner fits `log(price)`.

Exit ticket: the unrestricted tree has a training MAE of 0.66 EUR and a validation MAE of 3,053.52 EUR, while the tree limited to `min_samples_leaf=5` has 1,769.28 and 2,795.38. In one sentence, why does the tree with the larger training error predict unseen cars better?

In [ ]:
# Write your answer below. No Python is required.
#


# <font color='#E8800A'>References</font> <a class="anchor" id="references"></a>
[Back to TOC](#toc)

- Breiman, L., Friedman, J., Olshen, R. & Stone, C. (1984). *Classification and Regression Trees*. Wadsworth.
- Hastie, T., Tibshirani, R. & Friedman, J. (2009). *The Elements of Statistical Learning*, 2nd ed. Springer, chapter 9.2 (tree-based methods).
- James, G., Witten, D., Hastie, T. & Tibshirani, R. *An Introduction to Statistical Learning*, chapter 8.1 (the basics of decision trees).
- scikit-learn User Guide, [Decision Trees](https://scikit-learn.org/stable/modules/tree.html) · [`DecisionTreeRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html) · [Understanding the decision tree structure](https://scikit-learn.org/stable/auto_examples/tree/plot_unveil_tree_structure.html).